In [ ]:
#!pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html
import os
# Data agg
import pandas as pd
import numpy as np

# Viz
from matplotlib import pyplot as plt
import seaborn as sns
plt.style.use('seaborn')

# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.backends.cudnn.deterministic = True  

# metrics
from sklearn import metrics

# Data processing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

train = pd.read_csv('/opt/neurothink/neuro-243a1f02286547a78c6a147117496eba/train.csv')
test = pd.read_csv('/opt/neurothink/neuro-243a1f02286547a78c6a147117496eba/test.csv')

full_data = pd.concat([train,test],ignore_index=False)
missing_age_map = full_data.groupby(['Sex','Pclass','Parch','Embarked'],as_index=False).agg({'Age':'mean'})

train_age_merge = train.loc[train.Age.isnull(),:].drop(['Age'],axis=1).merge(missing_age_map,on=['Sex','Pclass','Parch','Embarked'],how='inner')
train_age_merge.index = train.loc[train.Age.isnull(),:].index
train.loc[train.Age.isnull(),'Age'] = train_age_merge['Age']

test_age_merge = test.loc[test.Age.isnull(),:].drop(['Age'],axis=1).merge(missing_age_map,on=['Sex','Pclass','Parch','Embarked'],how='inner')
test_age_merge.index = test.loc[test.Age.isnull(),:].index
test.loc[test.Age.isnull(),'Age'] = test_age_merge['Age']

train.Age.fillna(full_data.Age.mean(),inplace=True)
test.Age.fillna(full_data.Age.mean(),inplace=True)

train['Embarked'].fillna('S',inplace=True)
test['Embarked'].fillna('S',inplace=True)

train.columns = train.columns.str.lower().str.replace('/','_').str.replace(' ','_')
test.columns = test.columns.str.lower().str.replace('/','_').str.replace(' ','_')

# to drop columns
to_drop = ['name','ticket','cabin']
train = train.drop(to_drop + ['passengerid'],axis=1)
test = test.drop(to_drop,axis=1)

train.embarked = train.embarked.replace({'S': 0, 'C': 1, 'Q': 2})
test.embarked = test.embarked.replace({'S': 0, 'C': 1, 'Q': 2})


train.sex = train.sex.replace({'male':1,'female':0})
test.sex = test.sex.replace({'male':1,'female':0})

scaler = StandardScaler()

target = 'survived'

features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare','embarked']

train[features] = scaler.fit_transform(train[features])
test[features] = scaler.transform(test[features])

train_,test_ = train_test_split(train,test_size=0.33,random_state=42,stratify=train[target])
x_train = torch.from_numpy(train_[features].values).type(torch.FloatTensor)
y_train = torch.from_numpy(train_[target].values).type(torch.LongTensor)

x_test = torch.from_numpy(test_[features].values).type(torch.FloatTensor)
y_test = torch.from_numpy(test_[target].values).type(torch.LongTensor)

from datetime import datetime
now = datetime.now()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')
logfile.write(f"Start of Log\n")
logfile.write(f"Now training with device={device}\n")

### Here is our neural network
class Net(nn.Module):

    def __init__(self):
        super(Net,self).__init__()
        self.lin1 = nn.Linear(in_features=7,out_features=512,bias=True)
        self.lin2 = nn.Linear(in_features=512,out_features=512,bias=True)
        self.lin3 = nn.Linear(in_features=512,out_features=2,bias=True)
        self.activation = torch.nn.functional.relu
        #self.activation = torch.nn.functional.gelu
        self.dropout = torch.nn.Dropout(0.1)
    
    # here we take the input data and pass it through the chain of layers
    def forward(self,input):
        x = self.lin1(input)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.lin2(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.lin3(x)
        x = torch.nn.functional.sigmoid(x)
        return x

# instance our model
model = Net()
model.to(device)
# set the number of epochs
epochs = 100
# criterion aka loss function -> find more on pytorch docs
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# create 3 lists to store the losses and accuracy at each epoch
train_losses, test_losses, accuracy = [0]*epochs, [0]*epochs,[0]*epochs


# in this current case we don't use batches for training and we pass the whole data at each epoch
for e in range(epochs):
    optimizer.zero_grad()
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    
    # Comput train loss
    y_pred = model(x_train)
    loss = criterion(y_pred, y_train)
    
    loss.backward()

    optimizer.step()

    # store train loss
    train_losses[e] = loss.item()
    
    # Compute the test stats
    with torch.no_grad():
        # Turn on all the nodes
        model.eval()
        x_test = x_test.to(device)
        y_test = y_test.to(device)
        
        # Comput test loss
        ps = model(x_test)
        loss = criterion(ps, y_test)

        # store test loss
        test_losses[e] = loss.item()
        
        # Compute accuracy
        top_p, top_class = ps.topk(1, dim=1)
    
        equals = (top_class == y_test.view(*top_class.shape))
        
        # store accuracy
        accuracy[e] = torch.mean(equals.type(torch.FloatTensor))

# Print the final information
print(f'Accuracy  : {100*accuracy[-1].item():0.2f}%')
print(f'Train loss: {train_losses[-1]}')
print(f'Test loss : {test_losses[-1]}')
    
logfile.write(f'Accuracy  : {100*accuracy[-1].item():0.2f}%\n')
logfile.write(f'Train loss: {train_losses[-1]}\n')
logfile.write(f'Test loss : {test_losses[-1]}\n')
logfile.close()

# Plot the results

fig,ax = plt.subplots(1,2,figsize=(20,5))
ax[0].set_ylabel('Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_title('Model Accuracy')
ax[0].plot(accuracy)

ax[1].set_ylabel('Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_title('Train/Test Losses')
ax[1].plot(train_losses, label='train')
ax[1].plot(test_losses, label='test')
ax[1].legend()   

plt.tight_layout()
